In [84]:
import numpy as np 
import math 

In [85]:
data = np.array([
    [12.0, 1.5, 1, 'Wine'],
    [5.0, 2.0, 0, 'Beer'],
    [40.0, 0.0, 1, 'Whiskey'],
    [13.5, 1.2, 1, 'Wine'],
    [4.5, 1.8, 0, 'Beer'],
    [38.0, 0.1, 1, 'Whiskey'],
    [11.5, 1.7, 1, 'Wine'],
    [5.5, 2.3, 0, 'Beer'],
])

In [86]:
#Converting labels into integers
#Wine = 0 , Whiskey = 1 , Beer = 2
data[: , 3][data[:, 3] == 'Wine'] = 0
data[: , 3][data[:, 3] == 'Whiskey'] = 1
data[: , 3][data[:, 3] == 'Beer'] = 2

In [87]:
#Converting data into features and labels
X = np.array(data[:,0:3] , dtype = float)
y = np.array(data[:,3] , dtype = int)

In [114]:
#implementing the gini impurity function for a perticular set of labels
class Metric:
    def __init__(self , labels):
        self.labels = labels
    def entropy(self):
        values , counts = np.unique(self.labels , return_counts = True)
        total = np.sum(counts)
        sum_prob = 0
        for x in counts:
            if total!=0:
                prob = x/np.sum(counts)
                if x!=total:
                    sum_prob += -prob*(math.log2(prob)) - (1-prob)*(math.log2((total -x)/total))
        return (sum_prob)
    def gini(self):
        values , counts = np.unique(self.labels , return_counts = True)
        total = np.sum(counts)
        sum_prob = 0
        for x in counts:
            sum_prob+= x**2
        if total!=0:
            sum_prob = sum_prob/(total**2)
        return (1-sum_prob)
    def evaluate(self,metric):
        if metric == "gini" :
            return self.gini()
        elif metric == "entropy":
            return self.entropy()
        

In [115]:
def best_split(X,y, metric):
    samples , features = X.shape
    best_metric = np.array([])
    best_feature = np.array([])
    best_threshold = np.array([])
    for i in range (features):
        curr_feature = X[: , i]
        unique = np.unique(curr_feature)
        for x in unique:
            split1 = curr_feature <= x
            split2 = curr_feature > x
            y1 = y[split1]
            y2 = y[split2]
            metric1 = Metric(y1).evaluate(metric)
            metric2 = Metric(y2).evaluate(metric)
            
            weighted_metric = (len(y1)*metric1 + len(y2)*metric2) / samples
            best_metric = np.append(best_metric , weighted_metric)
            best_feature = np.append(best_feature , i)
            best_threshold = np.append(best_threshold , x)
            indices = np.argsort(best_metric)
            best_feature2 = best_feature[indices]
            best_threshold2 = best_threshold[indices]

            #now I will get the best feature , and the best threshold and the next threshold numerically greater to the best one 
            #reason explained in the loop function of DecisionTree class
            
            condition = best_feature2[:] == best_feature2[0]
            best_threshold2 = best_threshold2[condition]
            second_best = best_threshold2[:] > best_threshold2[0]
            arr = best_threshold2[second_best]
            np.sort(arr)
            if len(arr)!=0 :
                arr2 = np.append(np.array(best_threshold2[0]) , arr[0])
            else:
                arr2 = np.append(np.array(best_threshold2[0]) , best_threshold2[0])
    return int(best_feature2[0]) , arr2

In [116]:
class Node:
    def __init__(self , feature_index = 0 , threshold = np.array([]) , left = None ,right = None , value = None):
        self.feature_index = feature_index 
        self.threshold = threshold
        self.left = left 
        self.right = right 
        self. value = value     

In [117]:
class DecisionTree:
    def __init__(self, max_depth = float('inf') , threshold1= 1  , metric = "gini"):
        self.max_depth = max_depth
        self.threshold = threshold1
        self.parent_node = None
        self.metric = metric
    def fit(self,X,y):
        self.parent_node = self.train(X,y,0)
        
    def train(self , X ,y , depth):
        n_samples , n_features = X.shape
        value , counts = np.unique(y,return_counts = True)
        index = np.argmax(counts)
        max_value = value[index]
        
        if depth >= self.max_depth or len(np.unique(y)) == 1 or n_samples <=self.threshold :
            return Node(value = max_value)
            
        best_feature , best_threshold = best_split(X,y, self.metric)
        condition = X[: , best_feature] <= best_threshold[0]
        split1 , label1 = X[condition] , y[condition]
        split2 , label2 = X[~condition] , y[~condition]
        left_node = self.train(split1 , label1 , depth+1)
        right_node = self.train(split2 , label2 , depth +1)

        return Node(feature_index = best_feature , threshold = best_threshold , left = left_node , right = right_node)
    def loop(self , X , node:Node):
        if node.value is not None:
            return node.value
        if abs(X[node.feature_index] - node.threshold[0]) < abs(X[node.feature_index] - node.threshold[1]):
            #threshold[0[ stores the best threshold and threshold[1] stores the threshold above it. If the value 6 but the best
            #threshold is 5 , then the value will be misclassified if the nest threshold is very high. Therefore , here I compared the
            #difference in the value. If threshold[1] is 12 , then abs(5-6) < abs(12-6) therefore left node is called
            return self.loop(X , node.left)
        return self.loop(X,node.right) 

    def predict(self , X):
        predictions = np.array([])
        for x in range(X.shape[0]):
            current = X[x]
            value = self.loop(current , self.parent_node)
            predictions = np.append(predictions , value)
        return predictions
            

In [118]:
#Evaluation 
test_data = np.array([
    [6.0, 2.1, 0],   # Expected: Beer
    [39.0, 0.05, 1], # Expected: Whiskey
    [13.0, 1.3, 1]   # Expected: Wine
])
model_gini = DecisionTree(max_depth = float('inf') , threshold1 = 1 , metric = "gini")
model_gini.fit(X,y)
predictions = model_gini.predict(test_data)
print(np.array(predictions , dtype = int))

[2 1 0]


In [119]:
predictions = ["Wine" if x == 0 else x for x in predictions]
predictions = ["Whiskey" if x == 1 else x for x in predictions]
predictions = ["Beer" if x == 2 else x for x in predictions]
print(predictions)

['Beer', 'Whiskey', 'Wine']


In [120]:
model_entropy = DecisionTree(max_depth = float('inf') , threshold1 = 1 , metric = "entropy")
model_entropy.fit(X,y)
predictions = model_entropy.predict(test_data)
print(np.array(predictions , dtype = int))
predictions = ["Wine" if x == 0 else x for x in predictions]
predictions = ["Whiskey" if x == 1 else x for x in predictions]
predictions = ["Beer" if x == 2 else x for x in predictions]
print(predictions)

[2 1 0]
['Beer', 'Whiskey', 'Wine']
